# Asari & Zador (2009) on CRCNS ac-1

Paper: H. Asari and A. M. Zador (2009), "Long-Lasting Context Dependence Constrains Neural Encoding
Models in Rodent Auditory Cortex", J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009.

Dataset: CRCNS ac-1 (`crcns:ac-1/unversioned`, doi:10.6080/K0KW5CXR), read from the public bucket
crcnsarchive. CRCNS data sets are not versioned, so the version pinned is the one the bucket serves;
its `checksums.md5` is printed below for the three zip files read, so that a later change can be
detected. There is no other version to choose from (docs/archives.md, CRCNS).

Claims: `claims/asari-2009.yaml`, written from the version of record (text extraction of the
publisher's PDF); the online supplement was not supplied to the readers.

The data set ships the authors' MATLAB analysis code (`asari/Tools`: `CxtDep.m`, `GetEvoked.m`,
`GetSpikes.m`). The paper does not link it, but it is the authors' code for this analysis, and the
helper `asari.py` follows it where the Methods leave a detail open; each such place is marked in
the helper.

| Claim | Tested here |
|---|---|
| methods-recording-counts | tested (voltage-clamp counts not testable) |
| a1-mgb-firing-rates | tested |
| fig2-example-reliability | tested |
| fig2-example-long-context | tested |
| fig3a-proportion-significant | tested |
| fig3a-significance-decay | tested |
| fig3b-fractional-power-decay | tested |
| timescale-longer-than-membrane | tested |
| fig4-predictability-upper-bound | tested |
| fig4-strf-performance-low | blocked: not tested in this run (time) |
| fig5-example-conductances | blocked: data missing (no voltage-clamp recordings) |
| fig6-conductance-decay | blocked: data missing (no voltage-clamp recordings) |
| fig7-example-frequency-context | blocked: not tested in this run (time) |
| fig8a-proportion-by-property | blocked: not tested in this run (time) |
| fig8b-decay-by-property | blocked: not tested in this run (time) |
| fig8a-significance-fits | blocked: not tested in this run (time) |
| fig8c-relative-effects | blocked: not tested in this run (time) |
| fig8-natural-synthetic-probes-alike | blocked: not tested in this run (time) |
| fig9-mgb-fast-decay | tested |
| fig9-example-first-bin | tested |
| figs1-silence-probe | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time, resource, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import fsspec
import asari as A

DATASET = "crcns:ac-1/unversioned"
T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

## Data check

The bucket holds three zip files: `crcns-ac1.zip` (Wehr's data, Asari's stimuli and tools, and
`asari/Summary.mat`, which lists the cells and their files) and two zips of Asari's result files.
Every result file is opened here and its metadata read: recording type, unit, sampling rate,
length and trigger types.

In [ ]:
print(fsspec.open(A.BUCKET + "checksums.md5", "r").open().read())
rows = pd.DataFrame(A.summary())
print(rows.groupby(["experiment", "exp_description"]).agg(cells=("cell", "nunique"), files=("file", "size")))
survey = pd.DataFrame(A.run_threads(A.survey_file, sorted(A.members())))
print("result files:", len(survey), " GB:", round(survey["size"].sum() / 1e9, 2))
print("files listed in Summary.mat but missing from the zips:", sorted(set(rows.file) - set(survey.path)))
print("files in the zips not listed in Summary.mat:", sorted(set(survey.path) - set(rows.file)))
print(survey.groupby(["rectype", "unit", "sf"]).size())
types = collections.Counter()
for t in survey.trigger_types:
    types.update(t)
print("trigger types over all files:", dict(types))
print("files with 'pulse' triggers:",
      survey[survey.trigger_types.map(lambda t: "pulse" in t)][["path", "rectype", "unit", "description"]].to_string())

All 346 result files store a trace in mV at 10 kHz, recorded in whole-cell (current clamp) or
cell-attached mode. No file holds currents, and no experiment in `Summary.mat` is a voltage-clamp
one: the paper's 42 voltage-clamped cells (Figures 5 and 6) are not in the data set. 'Pulse'
triggers appear in two current-clamp files only.

The analysis sets follow `Summary.mat`: experiment 0 ("Auditory cortical cell responses to natural
sound combinations", 39 cells) for the natural sound current-clamp data, experiment 1 ("Thalamic
cell responses", 14 cells) for MGB, experiment 2 (synthetic sounds, 96 cells) for the synthetic
ensembles. Tuning-curve files are left out of the context analyses.

In [ ]:
nat_files = rows[(rows.experiment == 0) & (rows.description != "tuning curve")].file.tolist()
mgb_files = rows[(rows.experiment == 1) & (rows.description != "tuning curve")].file.tolist()
syn_files = rows[(rows.experiment == 2) & (rows.description != "tuning curve")].file.tolist()
print("natural A1 files:", len(nat_files), " MGB files:", len(mgb_files), " synthetic A1 files:", len(syn_files))
sv = survey.set_index("path")
for name, fl in [("natural", nat_files), ("MGB", mgb_files), ("synthetic", syn_files)]:
    s = sv.loc[fl]
    print(name, dict(s.rectype.value_counts()), "units", set(s.unit), "sf", set(s.sf),
          "minutes per file %.1f to %.1f" % (s.n.min() / 6e5, s.n.max() / 6e5),
          "animals", len({A.animal_id(f) for f in fl}), "cells", len({A.cell_id(f) for f in fl}))
rec_x = A.Recording("Results/20051107-ha1-001/20051107-ha1-001-003.mat")
print("20051107-ha1-001-003: table rows", rec_x.table, "; trigger labels", sorted({t["description"] for t in rec_x.triggers if t["type"] != "pulse"}))
dup = sv.loc[nat_files + mgb_files + syn_files].groupby(["n", "size"]).size()
print("file pairs with identical length and size (possible double deposits):", int((dup > 1).sum()))

### Running the analyses

Each natural sound file is analysed in a worker process (`A.analyse_a1_file`): the trace is median
filtered (10 ms), cut into probe sets as `CxtDep.m` does, and for each probe the Kruskal-Wallis
significance measure (1,000 shuffles for the noise floor), the power terms of Eqs. 17 and 18, the
trial-to-trial correlations and the noise autocorrelation are computed, with all contexts and
without "silence". Spikes are counted by period. MGB files go through `A.analyse_mgb_file`
(spikes, PSTHs). This takes about six minutes.

In [ ]:
t = time.time()
NAT = A.run_parallel(A.analyse_a1_file, nat_files)
print("natural files analysed in %.0f s" % (time.time() - t))
t = time.time()
MGB = A.run_parallel(A.analyse_mgb_file, mgb_files)
print("MGB files analysed in %.0f s" % (time.time() - t))
print("peak memory of this process: %.2f GB" % (resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

In [ ]:
# trial table checks: repeats, fragments, and the trigger clock
chk = pd.DataFrame([dict(file=A.cell_id(f["path"]) + "/" + f["path"].split("/")[-1][-7:-4], rep=f["rep"],
                         fragments=f["n_frag"], minutes=round(f["n_samples"] / f["sf"] / 60, 1),
                         vm_p1=round(f["vrange"][0], 1), vm_p99=round(f["vrange"][1], 1),
                         corr_aligned=round(f["clock"][0], 3), corr_shifted=round(f["clock"][1], 3))
                    for f in NAT])
print(chk.to_string())
print("aligned > shifted in %d of %d files; median aligned %.3f, shifted %.3f"
      % ((chk.corr_aligned > chk.corr_shifted).sum(), len(chk), chk.corr_aligned.median(), chk.corr_shifted.median()))
mg = pd.DataFrame([dict(file=A.cell_id(g["path"]) + "/" + g["path"].split("/")[-1][-7:-4], rep=g["rep"],
                        fragments=g["n_frag"], onsets=g["clock"][2], spikes_100ms_after_onset=g["clock"][0],
                        spikes_100ms_before_onset=g["clock"][1]) for g in MGB])
print(mg.to_string())
more = mg.spikes_100ms_after_onset > mg.spikes_100ms_before_onset
print("MGB files with more spikes after onsets than before: %d of %d; not: %s" % (more.sum(), len(mg), mg.file[~more].tolist()))

The median-filtered membrane potential has 1st percentiles from -84 to -43 mV and 99th
percentiles from -54 to -4 mV over the files, values expected for whole-cell recordings in mV. The correlation of the responses to the first sequence across repeats
is higher when the repeats are aligned on their triggers than when each is shifted by a different
lag, in the files listed above: the responses are locked to the trigger times, so the traces and
the trigger table share a clock. In MGB, spikes are more frequent in the 100 ms after stimulus
onsets than in the 100 ms before in 18 of 21 files, consistent with the triggers marking sound
onsets. In the other three (cells 20071012-ha1-002 and 20071012-ha1-003) there are fewer spikes
after onsets than before, so this contrast does not establish the clock for those two cells; their
probes are kept in the MGB analyses.

## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with natural
> sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated colored
> noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

A probe counts when it has at least two conditioning stimuli each with at least 4 trials (current
clamp) or 10 trials (MGB) (Methods, Context dependence at subthreshold voltage level; Table 1
legend). Readings: silence counted as a conditioning stimulus, or not. Synthetic files are cut into
probe sets in the same way (all contexts) for the cell count.

In [ ]:
def nat_probes(key="all"):
    out = []
    for f in NAT:
        for p in f["probes"]:
            if p["probe"] > 0 and p["m"] >= 4 and (key == "all" or p["ncx_nosil"] >= 2):
                out.append((f, p))
    return out

for key in ["all", "nosil"]:
    P = nat_probes(key)
    print("natural, silence %s: %d probes in %d cells, %d animals" % (
        "counted" if key == "all" else "not counted", len(P), len({A.cell_id(f["path"]) for f, p in P}),
        len({A.animal_id(f["path"]) for f, p in P})))
print("trials per context (m) over natural probes:", dict(collections.Counter(p["m"] for f, p in nat_probes())))
print("natural probes per file design:", dict(collections.Counter(f["n_frag"] for f, p in nat_probes())))
for key, rule in [("all", 10)]:
    P = [(g, p) for g in MGB for p in g["probes"] if p["m"] >= rule]
    print("MGB: %d probes with >= 10 trials per context in %d cells (%d probes in all files); trials per context %s"
          % (len(P), len({A.cell_id(g["path"]) for g, p in P}), sum(len(g["probes"]) for g in MGB),
             dict(collections.Counter(p["m"] for g, p in P))))
    P0 = [(g, p) for g, p in P if len([c for c in p["contexts"] if c != 0]) >= 2]
    print("MGB, silence not counted: %d probes in %d cells" % (len(P0), len({A.cell_id(g["path"]) for g, p in P0})))

def syn_count(path):
    rec = A.Recording(path)
    ps = A.probe_sets(rec, rec.trace)
    return dict(path=path, description=rec.description, rep=rec.rep,
                stimuli=sorted({s["description"][:24] for s in rec.stimuli}),
                probes_m4=sum(q["m"] >= 4 for q in ps),
                probes_m4_nosil=sum(q["m"] >= 4 and len([c for c in q["contexts"] if c != 0]) >= 2 for q in ps))
SYN = pd.DataFrame(A.run_threads(syn_count, syn_files))
SYN["cell"] = SYN.path.map(A.cell_id)
print(SYN.groupby("description").agg(files=("path", "size"), cells=("cell", "nunique"), probes=("probes_m4", "sum")).to_string())
syn_cells = set(SYN[SYN.probes_m4 > 0].cell)
nat_cells = {A.cell_id(f["path"]) for f, p in nat_probes()}
print("synthetic cells with a probe meeting the criterion: %d of %d; cells also in the natural set: %d"
      % (len(syn_cells), SYN.cell.nunique(), len(syn_cells & nat_cells)))
print("A1 current-clamp cells meeting the criterion (natural or synthetic): %d" % len(syn_cells | nat_cells))
print("excluding the files of 'naturalistic' sounds and of a natural probe after natural and pink-noise contexts "
      "(no Table 1 column): %d" % len(nat_cells | set(SYN[(SYN.probes_m4 > 0) & ~SYN.description.str.contains(
          "naturalistic|6 natural sound conditioning")].cell)))

**Result: partially reproduced; the natural sound and MGB counts match the paper exactly, the 123 cells do not, and the voltage-clamp counts cannot be tested.** There are 305 natural sound probes in 39 A1 cells whether or not silence counts as a conditioning stimulus. There are 93 MGB probes in 14 cells. 132 A1 current-clamp cells meet the criterion (39 natural, 95 synthetic, 2 in both), against the paper's 123. Leaving out the two synthetic designs that have no column in Table 1 gives 124. The voltage-clamp counts (29 probes, 14 cells) cannot be tested, as the data set holds no voltage-clamp recordings.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD,
> 194 cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

Rates per cell over its context files; spontaneous over the intersequence silences or the 1 s
before each sequence (baseline), evoked over the sound periods. A1 spikes are excursions of the raw
trace above its median-clipped trace (5, 10 or 20 mV, an assumption); MGB spikes on the code's
high-pass filtered trace, with the code's automatic threshold or 20 or 40 robust SDs. The data set
holds 39 natural sound A1 cells and 14 MGB cells with context files, not the paper's 194 and 16.
The probe periods analysed cover every sound fragment, so the second evoked reading equals the first.

In [ ]:
cell_of = {f["path"]: A.cell_id(f["path"]) for f in NAT}
def cell_rates(files, key, spont):
    acc = {}
    for f in files:
        r = f["rates"][key]
        a = acc.setdefault(A.cell_id(f["path"]), np.zeros(4))
        a += [r[spont][0], r[spont][1], r["sound"][0], r["sound"][1]]
    s = np.array([v[0] / v[1] for v in acc.values()])
    e = np.array([v[2] / v[3] for v in acc.values()])
    return len(acc), s, e

expect = {"A1": (0.47, 0.61, 0.57, 0.77), "MGB": (0.78, 1.25, 11.4, 16.9)}
tab = []
for area, files, keys in [("A1", NAT, [5.0, 10.0, 20.0]), ("MGB", MGB, list(A.MGB_THRESHOLDS))]:
    for key in keys:
        for spont in ["silence", "baseline"]:
            n, s, e = cell_rates(files, key, spont)
            ex = expect[area]
            tab.append(dict(area=area, threshold=key, spontaneous_window=spont, cells=n,
                            spont_mean=s.mean(), spont_sd=s.std(ddof=1), evoked_mean=e.mean(), evoked_sd=e.std(ddof=1),
                            spont_ok=abs(s.mean() - ex[0]) <= 0.1 * ex[0], evoked_ok=abs(e.mean() - ex[2]) <= 0.1 * ex[2],
                            spont_max=s.max(), evoked_max=e.max()))
tab = pd.DataFrame(tab)
print(tab.round(3).to_string())
a1ok = tab[tab.area == "A1"]
mgok = tab[tab.area == "MGB"]
print("A1 readings with both means within 10%:", int((a1ok.spont_ok & a1ok.evoked_ok).sum()), "of", len(a1ok))
print("MGB readings with both means within 10%:", int((mgok.spont_ok & mgok.evoked_ok).sum()), "of", len(mgok))
print("A1 means below 1 Hz in every reading:", bool((a1ok[["spont_mean", "evoked_mean"]] < 1).all().all()))
print("MGB evoked mean above 5 Hz:", dict(zip(mgok.threshold.astype(str) + "/" + mgok.spontaneous_window, (mgok.evoked_mean > 5))))
print("MGB spike thresholds (mV on the filtered trace), median over files:",
      {k: round(float(np.median([g["threshold"][k] for g in MGB])), 3) for k in A.MGB_THRESHOLDS})

**Result: partially reproduced; no reading puts all four means within 10%, but A1 rates are below 1 Hz and MGB evoked rates are well above them.** No A1 reading has both means within tolerance: with a 10 mV threshold the evoked mean matches but the spontaneous mean is higher, and with 20 mV both are lower. No MGB reading has both. A1 means are below 1 Hz in every reading. The MGB evoked mean is above 5 Hz with the 20 and 40 SD thresholds, but not with the code's automatic threshold (5.0 Hz), which is set 5 to 9 times higher.

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (correlation across trials
> 0.74 +/- 0.08 for the shown probe and 0.61 +/- 0.07 over the seven natural sound fragments tested),
> and the mean correlation over the population was 0.31 +/- 0.09.

Readings: (a) mean correlation over pairs of trials, (b) each trial against the mean of the others;
population mean over cells, or over probes. Correlations are over the probe period, within each
context, then averaged over contexts.

In [ ]:
P = nat_probes()
df = pd.DataFrame([dict(cell=A.cell_id(f["path"]), animal=A.animal_id(f["path"]), file=f["path"], probe=p["probe"],
                        a=p["corr"][0], b=p["corr"][1], n_frag=f["n_frag"], rep=f["rep"], contexts=p["contexts"])
                   for f, p in P])
for r in ["a", "b"]:
    per_cell = df.groupby("cell")[r].mean()
    print("reading %s: over probes %.3f +/- %.3f (n=%d); over cells %.3f +/- %.3f (n=%d)"
          % (r, df[r].mean(), df[r].std(ddof=1), len(df), per_cell.mean(), per_cell.std(ddof=1), len(per_cell)))
print("range over cells, reading a: %.2f to %.2f" % (df.groupby("cell").a.mean().min(), df.groupby("cell").a.mean().max()))
cand = df[(df.n_frag == 7) & (df.rep == 6)]
print("cells with seven natural fragments and six repeats:")
print(cand.groupby("cell").agg(a=("a", "mean"), a_sd=("a", "std"), b=("b", "mean"), probes=("probe", "size")).round(3))
print(cand[["cell", "probe", "contexts", "a", "b"]].round(3).to_string())

**Result: reproduced; the population mean is 0.31 under reading (a), and cell 20051222-ha1-001 matches the paper's example values.** Under reading (a), the mean correlation over cells is 0.31 (SD 0.09) and over probes 0.32. Reading (b) gives higher values, outside the tolerance. Of the five cells with seven natural fragments and six repeats, 20051222-ha1-001 has a mean of 0.61 over its seven fragments, and its probe 4 (contexts silence, 2 and 3) has 0.74. These are the paper's example values, so it is taken as the example cell.

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe for
> about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again after
> about 4 s.

The claim was amended after a question to a reader (`questions/asari-2009.yaml`). The example is
matched by the reliability values of the Figure 2A legend: a current-clamp cell with seven natural
sound fragments whose mean reliability (reading (a) of fig2-example-reliability) is within
0.61 +/- 0.07, and in it a probe tested after silence and two other fragments, with six repeats in at
least one context, whose reliability is within 0.74 +/- 0.08; the closest to both values where
several match. Reproduced when the matched probe has significant periods before 2 s and after
3.6 s and none between 2 and 3.6 s. The strict reading (none between 2 and 4 s) is reported, and
every probe with seven fragments, silence and two fragments is shown.


In [ ]:
def runs_s(p, key="all"):
    return [(round(a / 1e4, 3), round(b / 1e4, 3)) for a, b in p[key]["runs"]]
# matching rule of the amended criterion (Figure 2A legend values)
cell_rel = df[df.n_frag == 7].groupby("cell").a.mean()
print("cells with seven natural fragments, mean reliability (reading a):")
print(cell_rel.round(3).to_string())
match = []
for f, p in P:
    cid = A.cell_id(f["path"])
    if f["n_frag"] != 7 or abs(cell_rel[cid] - 0.61) > 0.07:
        continue
    ok_ctx = len(p["contexts"]) == 3 and 0 in p["contexts"]
    six = max(p["n_occ"].values()) >= 6
    match.append(dict(cell=cid, cell_rel=round(cell_rel[cid], 3), probe=p["probe"], contexts=p["contexts"],
                      six_repeats=six, probe_rel=round(p["corr"][0], 3),
                      matches=ok_ctx and six and abs(p["corr"][0] - 0.74) <= 0.08,
                      distance=round(abs(cell_rel[cid] - 0.61) + abs(p["corr"][0] - 0.74), 3)))
match = pd.DataFrame(match)
print("probes of cells within 0.61 +/- 0.07:")
print(match.to_string())
best = match[match.matches].sort_values("distance").iloc[0]
print("matched probes: %d; closest: %s probe %d" % (match.matches.sum(), best.cell, best.probe))
f_ex = [f for f, p in P if A.cell_id(f["path"]) == best.cell][0]
p_ex = [p for p in f_ex["probes"] if p["probe"] == best.probe][0]

def window_test(r, lo, hi):
    return dict(before_2s=any(a < 2 for a, b in r), after_hi=any(b > hi for a, b in r),
                none_in_window=not any(b > lo and a < hi for a, b in r),
                overlap_ms=round(1000 * sum(max(0, min(b, hi) - max(a, lo)) for a, b in r), 1))
r = runs_s(p_ex)
print("matched probe, significant periods (s), all contexts:", [(float(a), float(b)) for a, b in r])
print("main reading, 2 to 3.6 s:", window_test(r, 2.0, 3.6))
print("strict reading, 2 to 4 s:", window_test(r, 2.0, 4.0))
gap = [(float(b0), float(a1)) for (a0, b0), (a1, b1) in zip(r[:-1], r[1:]) if a1 - b0 > 1]
print("gaps longer than 1 s (s):", gap)
print("same probe without silence:", [(float(a), float(b)) for a, b in runs_s(p_ex, "nosil")])
rowsx = []
for f, p in P:
    if f["n_frag"] == 7 and len(p["contexts"]) == 3 and 0 in p["contexts"]:
        rr = runs_s(p)
        w36, w4 = window_test(rr, 2.0, 3.6), window_test(rr, 2.0, 4.0)
        rowsx.append(dict(cell=A.cell_id(f["path"]), probe=p["probe"], T_s=p["T"] / 1e4, n_periods=len(rr),
                          before_2s=w36["before_2s"], after_3_6s=w36["after_hi"], none_2_3_6=w36["none_in_window"],
                          after_4s=w4["after_hi"], none_2_4=w4["none_in_window"], overlap_2_4_ms=w4["overlap_ms"]))
print("all probes with seven fragments and contexts silence plus two fragments:")
print(pd.DataFrame(rowsx).to_string())
late = [(A.cell_id(f["path"]), p["probe"]) for f, p in P if any(b > 4e4 for a, b in p["all"]["runs"])]
print("natural probes with a significant period later than 4 s: %d of %d" % (len(late), len(P)))


**Result: reproduced under the amended criterion; the strict 2 to 4 s reading fails by 30 ms.** Only one probe matches the Figure 2A values: probe 4 of 20051222-ha1-001 (cell mean 0.611, probe 0.737). No other cell with seven fragments has a mean within 0.61 +/- 0.07. In this probe, context dependence is significant from 0.03 s to 1.60 s and again from 3.97 s to 4.60 s, with nothing significant between 1.60 and 3.97 s. So there are significant periods before 2 s and after 3.6 s, and none between 2 and 3.6 s. Under the strict reading, the period starting at 3.97 s overlaps the 2 to 4 s interval by 30 ms. Without silence among the contexts, the probe has only two short significant periods. 17 of 305 natural probes have a significant period later than 4 s.


## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound
> contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of the
> significant events occurred more than 1 s after probe onset.

Readings: silence counted among the contexts or not; the proportion at time t over all 305 probes,
or over the probes still running at t. The late fraction is one minus the cumulative probability at
1 s, the cumulative being the running integral of the proportion less the noise floor, normalised at
its peak (Figure 3A legend).

In [ ]:
def population(key, denom):
    PP = nat_probes(key)
    L = max(p["T"] for f, p in PP) // 10
    S, R, NZ = np.zeros(L), np.zeros(L), np.zeros(L)
    for f, p in PP:
        s = np.zeros(p["T"], bool)
        for a, b in p[key]["runs"]:
            s[a:b] = True
        sb = A.bin_mean(s.astype(float), 10)
        S[:len(sb)] += sb
        R[:len(sb)] += 1
        NZ[:len(p[key]["noise"])] += p[key]["noise"]
    den = len(PP) if denom == "all" else np.maximum(R, 1)
    return np.arange(L) * 1e-3, S / den, NZ / den, R, PP

FIT3A = {}
res = []
for key in ["all", "nosil"]:
    for denom in ["all", "running"]:
        t, prop, nf, R, PP = population(key, denom)
        nsig = sum(len(p[key]["runs"]) > 0 for f, p in PP)
        c = np.cumsum(prop - nf)
        ipk = int(np.argmax(c))
        cum = c / c[ipk]
        FIT3A[(key, denom)] = (t, prop, nf)
        res.append(dict(silence=("counted" if key == "all" else "not counted"), denominator=denom, probes=len(PP),
                        significant=nsig, fraction=nsig / len(PP), cum_peak_s=t[ipk], late_fraction=1 - cum[1000],
                        noise_floor_mean_0_6s=nf[t < 6].mean()))
res = pd.DataFrame(res)
res["count_ok"] = (res.significant - 204).abs() <= 20.4
res["fraction_ok"] = (res.fraction - 0.669).abs() <= 0.0669
res["late_ok"] = (res.late_fraction - 0.237).abs() <= 0.0237
print(res.round(4).to_string())
df_sig = pd.DataFrame([dict(animal=A.animal_id(f["path"]), cell=A.cell_id(f["path"]), sig=len(p["all"]["runs"]) > 0)
                       for f, p in nat_probes()])
by_animal = df_sig.groupby("animal").sig.agg(["sum", "size", "mean"])
print("by animal (silence counted): %d animals; fraction significant per animal median %.2f, range %.2f to %.2f"
      % (len(by_animal), by_animal["mean"].median(), by_animal["mean"].min(), by_animal["mean"].max()))
print(by_animal.T.to_string())
t, prop, nf = FIT3A[("all", "all")]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(t, prop, "k", lw=1, label="proportion significant")
ax.plot(t, nf, color="0.6", label="noise floor")
ax.set_xlim(0, 6); ax.set_xlabel("time from probe onset (s)"); ax.set_ylabel("probability"); ax.legend()
print("plotted range: proportion %.3f to %.3f over 0 to %.2f s; none outside the axes up to 6 s" % (prop.min(), prop.max(), t[-1]))

**Result: reproduced; with silence counted and the proportion taken over all 305 probes, all three values are within tolerance.** 186 of 305 probes (61.0%) are significant, the late fraction is 0.252 and the cumulative curve peaks at 4.78 s (paper 4.72 s). Over the probes still running, the late fraction is 0.267, outside tolerance. With silence left out of the contexts, 166 probes (54.4%) are significant and the late fraction is 0.17 to 0.18, both outside tolerance. The other rows of the table show these readings.

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the sum
> of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise floor of
> about 2.8 x 10^-3.

Eq. 44 with the constant fixed at the mean noise floor, fitted from 0 to 4.72 s or 0 to 6 s, with
the paper's rule for the number of exponentials. Sizes are fitted with free sign, or bounded at zero
(the paper calls them decay sizes); both are shown.

In [ ]:
out = []
for (key, denom), (t, prop, nf) in FIT3A.items():
    kappa = nf[t < 6].mean()
    for t_end in [4.72, 6.0]:
        sel = t <= t_end
        for nonneg in [False, True]:
            k, fits = A.fit_exp_select(t[sel], prop[sel], kappa=kappa, nonneg=nonneg)
            f2 = fits[2]
            out.append(dict(silence=key, denominator=denom, fit_to=t_end, sizes=("nonneg" if nonneg else "free"),
                            n_selected=k, kappa=kappa, a1=f2["a"][0], tau1=f2["tau"][0], a2=f2["a"][1], tau2=f2["tau"][1],
                            one_exp_a=fits[1]["a"][0], one_exp_tau=fits[1]["tau"][0]))
out = pd.DataFrame(out)
exp3a = dict(a1=(0.17, 0.017), tau1=(0.20, 0.02), a2=(0.09, 0.009), tau2=(0.90, 0.09))
for c, (v, tol) in exp3a.items():
    out[c + "_ok"] = (out[c] - v).abs() <= tol + 1e-12
out["kappa_ok"] = (out.kappa - 0.0028).abs() <= 0.00028
out["partial"] = (out.n_selected == 2) & out.tau2.between(0.5, 2)
print(out.round(4).to_string())

**Result: partially reproduced, resting on the assumption that sizes are non-negative; no reading puts every value within tolerance.** With sizes bounded at zero and silence counted, two exponentials are selected in every reading. The slow constant is 0.97 to 1.08 s, within tolerance in one reading (all probes, fit to 6 s). The fast constant is 0.25 to 0.27 s and the slow size 0.068 to 0.080, both outside tolerance. The fast size is 0.149 to 0.160, within tolerance when the proportion is over the probes still running. With free signs, the fit spends one exponential on the rise of the curve after probe onset (a negative size), and the slow constant is 0.44 s, below the 0.5 s of the partial rule. Without silence, one exponential is selected. The mean noise floor is 2.8 x 10^-3 over the probes still running, and 2.2 x 10^-3 over all probes.

## fig3b-fractional-power-decay and fig4-predictability-upper-bound

> fig3b: The context-dependent fraction of the stimulus-related response power, normalized by the
> smoothed stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

> fig4: Without knowledge of the context, no model can capture more than about half (0.51) of the
> predictable response power with a window of about 100 ms; the upper bound approaches one only
> with a window of seconds (time constant 1.04 s).

Eq. 17 (stimulus-related) and Eq. 18 (context-independent) are averaged over probes at each time
point; Q(t) of Eq. 20 is (Eq. 17 - Eq. 18) over the running mean of Eq. 17 over [0, 2t]; the upper
bound of Eq. 43 is Eq. 18 over the same smoothed denominator (main reading) or over the unsmoothed
one (variant). Readings: silence counted or not, average over all probes or over the probes
running, fit to 4 s or 6 s.

In [ ]:
POW = {}
out = []
for key in ["all", "nosil"]:
    PP = nat_probes(key)
    L = max(len(p[key]["pred"]) for f, p in PP)
    t = np.arange(L) * 1e-3
    PR, IV, R = np.zeros(L), np.zeros(L), np.zeros(L)
    for f, p in PP:
        n = len(p[key]["pred"])
        PR[:n] += p[key]["pred"]; IV[:n] += p[key]["inv"]; R[:n] += 1
    for denom in ["all", "running"]:
        den = len(PP) if denom == "all" else np.maximum(R, 1)
        pr, iv = PR / den, IV / den
        sm = A.running_mean_0_2t(pr, 1e-3)
        Q, U, U0 = (pr - iv) / sm, iv / sm, iv / pr
        POW[(key, denom)] = (t, pr, iv, sm, Q, U)
        for t_end in [4.0, 6.0]:
            sel = t <= t_end
            fq = A.fit_exp(t[sel], Q[sel], 1, kappa=0.0)
            fu = A.fit_exp(t[sel], U[sel], 1, kappa=1.0)
            f0 = A.fit_exp(t[sel], U0[sel], 1, kappa=1.0)
            out.append(dict(silence=key, denominator=denom, fit_to=t_end, Q_a1=fq["a"][0], Q_tau1=fq["tau"][0],
                            U_1plus_a1=1 + fu["a"][0], U_tau1=fu["tau"][0],
                            U_unsmoothed_1plus_a1=1 + f0["a"][0], U_unsmoothed_tau1=f0["tau"][0],
                            Q_at_0=Q[0], Q_at_1s=Q[1000], U_at_0_1s=U[100]))
out = pd.DataFrame(out)
out["fig3b_ok"] = ((out.Q_a1 - 0.49).abs() <= 0.049) & ((out.Q_tau1 - 1.04).abs() <= 0.104)
out["fig4_ok"] = ((out.U_1plus_a1 - 0.51).abs() <= 0.051) & ((out.U_tau1 - 1.04).abs() <= 0.104)
out["fig4_unsmoothed_ok"] = ((out.U_unsmoothed_1plus_a1 - 0.51).abs() <= 0.051) & ((out.U_unsmoothed_tau1 - 1.04).abs() <= 0.104)
print(out.round(3).to_string())
t, pr, iv, sm, Q, U = POW[("all", "all")]
print("stimulus-related power (mV^2) at 0, 0.1, 1, 4 s: %s; smoothed: %s" % (pr[[0, 100, 1000, 3999]].round(2), sm[[0, 100, 1000, 3999]].round(2)))
print("Q range over 0-6 s: %.3f to %.3f; U range: %.3f to %.3f" % (Q[t < 6].min(), Q[t < 6].max(), U[t < 6].min(), U[t < 6].max()))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
sel = t < 6
ax[0].plot(t[sel], Q[sel], "k", lw=0.8); ax[0].set_title("Q(t), Eq. 20"); ax[0].set_ylim(-0.2, 1.2)
ax[1].plot(t[sel], U[sel], "k", lw=0.8); ax[1].set_title("upper bound, Eq. 43"); ax[1].set_ylim(-0.2, 1.2)
for a_ in ax: a_.set_xlabel("time from probe onset (s)")
print("points outside the plotted range -0.2 to 1.2: Q %d, U %d" % (((Q[sel] < -0.2) | (Q[sel] > 1.2)).sum(), ((U[sel] < -0.2) | (U[sel] > 1.2)).sum()))

**Result: fig3b-fractional-power-decay is not reproduced; the size matches, but the time constant is longer than 1.04 s in every reading.** The size of Q(t) is 0.45 to 0.47, within tolerance. The time constant is 1.3 to 1.8 s.

**Result: fig4-predictability-upper-bound is not reproduced; the upper bound near onset is close to one half, but the time constant is longer than 1.04 s in every reading.** With the smoothed denominator, 1 + alpha1 is 0.43 to 0.49 in the fits that stay within bounds, within tolerance in three of them. Their time constant is 1.66 to 2.81 s; the two fits to 6 s over all probes run to the fit's 100 s bound. With the unsmoothed denominator, 1 + alpha1 is 0.50 to 0.54, within tolerance in every reading, and the time constant is 1.46 to 1.92 s.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay
> within about 100 ms, much faster than the second-long timescale of context dependence in A1.

Noise is each trial less its context mean (Eq. 14); spontaneous activity is the intersequence
silences (whole, or without their first 1 s), median filtered as for the responses. The
autocorrelation is normalised by the variance (Eq. 45) and averaged.

In [ ]:
def lag(ac, th):
    i = np.where(ac < th)[0]
    return i[0] * 1e-3 if len(i) else np.nan
nac = np.mean([p["noise_ac"] for f, p in nat_probes()], axis=0)
sp = [f["spont_ac"] for f in NAT if f["spont_ac"] is not None]
sp1 = [f["spont_ac_skip1s"] for f in NAT if f["spont_ac_skip1s"] is not None]
sac, sac1 = np.mean(sp, axis=0), np.mean(sp1, axis=0)
tl = []
for name, ac, n in [("noise", nac, len(nat_probes())), ("spontaneous", sac, len(sp)), ("spontaneous, first 1 s left out", sac1, len(sp1))]:
    tl.append(dict(series=name, n=n, lag_1_over_e_s=lag(ac, 1 / np.e), lag_0_1_s=lag(ac, 0.1)))
tl = pd.DataFrame(tl)
print(tl.to_string())
per_file = [lag(np.mean([p["noise_ac"] for p in f["probes"] if "noise_ac" in p and p["m"] >= 4], axis=0), 1 / np.e) for f in NAT]
print("noise 1/e lag per file: median %.3f s, range %.3f to %.3f s" % (np.median(per_file), np.min(per_file), np.max(per_file)))
print("all 1/e and 0.1 lags under 150 ms:", bool((tl[["lag_1_over_e_s", "lag_0_1_s"]] < 0.15).all().all()),
      "; 1/e lags below 1.04/3 s:", bool((tl.lag_1_over_e_s < 1.04 / 3).all()))

**Result: reproduced; both autocorrelations decay within about 0.1 s, much faster than the second-long context dependence.** The noise autocorrelation falls below 1/e at 0.07 s and below 0.1 at 0.11 s. The spontaneous autocorrelation does the same at 0.08 and 0.12 s. All lags are under 150 ms and far below a third of 1.04 s.

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the intersequence
> silences, but the response power attributable to the preceding stimulus alone faded within about
> 1 s after the stimulus ended.

The silence probe is the first 2.5 s of each intersequence silence (`CxtDep.m` default, cut to the
shortest silence), with the last fragment of each sequence as its context; at least 4 trials per
context.

In [ ]:
SP = [(f, p) for f in NAT for p in f["probes"] if p["probe"] == -1 and p["m"] >= 4]
starts = [(A.cell_id(f["path"]), p["all"]["runs"][0][0] / 1e4) for f, p in SP if len(p["all"]["runs"])]
early_cells = sorted({c for c, s0 in starts if s0 < 0.5})
print("silence probes: %d in %d cells; with a significant period: %d; cells with one starting within 0.5 s: %d"
      % (len(SP), len({A.cell_id(f["path"]) for f, p in SP}), len(starts), len(early_cells)))
L = min(len(p["all"]["pred"]) for f, p in SP)
t = np.arange(L) * 1e-3
pr = np.mean([p["all"]["pred"][:L] for f, p in SP], axis=0)
cd = np.mean([p["all"]["pred"][:L] - p["all"]["inv"][:L] for f, p in SP], axis=0)
for name, y in [("stimulus-related power (Eq. 17)", pr), ("context-dependent power (Eq. 19)", cd)]:
    y10 = A.bin_mean(y, 10)
    t10 = np.arange(len(y10)) * 0.01
    below = np.where(y10 < 0.1 * y10[0])[0]
    print("%s: %.2f mV^2 at onset (first 10 ms), %.3f at 1 s, %.3f at 2 s; first below 10%% of onset at %s s"
          % (name, y10[0], y10[100], y10[200], round(t10[below[0]], 2) if len(below) else "never"))
print("silence probe length used: %.2f s" % (L / 1000))

**Result: reproduced; 19 cells show significance soon after the silence begins, and the power fades within 1 s.** There are 44 silence probes in 38 cells. 19 cells have a significant period starting within 0.5 s of the silence onset. The stimulus-related power over silence probes (Eq. 17) falls below 10% of its onset value at 0.72 s, and the context-dependent power (Eq. 19) at 0.46 s, both before 1.1 s.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short time
> (time constant about 80 ms; 93 probes in 14 cells), so the long-lasting component in A1 arises
> mainly in cortex.

PSTHs in 100-ms bins for k = 0 to 60, SD over contexts (n - 1), averaged over the 93 probes; one
exponential with a free constant. Readings: bin time at its start or centre; spike threshold; the
average over all probes or over probes running.

In [ ]:
MP = [(g, p) for g in MGB for p in g["probes"] if p["m"] >= 10]
out = []
for key in A.MGB_THRESHOLDS:
    sds = [p["psth_%s" % key].std(axis=0, ddof=1)[:61] for g, p in MP]
    S, R = np.zeros(61), np.zeros(61)
    for x in sds:
        S[:len(x)] += x; R[:len(x)] += 1
    for denom in ["all", "running"]:
        y = S / (len(MP) if denom == "all" else np.maximum(R, 1))
        for tb, off in [("start", 0.0), ("centre", 0.05)]:
            tt = np.arange(61) * 0.1 + off
            f = A.fit_exp(tt, y, 1, None)
            out.append(dict(threshold=key, denominator=denom, bin_time=tb, a1=f["a"][0], tau1=f["tau"][0],
                            kappa=f["kappa"], mean_after_0_5s=y[tt > 0.5].mean(), first_bins=np.round(y[:3], 2)))
out = pd.DataFrame(out)
out["tau_ok"] = ((out.tau1 - 0.08).abs() <= 0.008) | (out.tau1 < 0.1)
out["plateau_vs_fit"] = (out.mean_after_0_5s - out.kappa).abs() <= 0.25 * out.kappa.abs()
out["plateau_vs_2"] = (out.mean_after_0_5s - 2.0).abs() <= 0.5
print(out.round(3).to_string())
print("probes per cell:", dict(collections.Counter(A.cell_id(g["path"]) for g, p in MP)))

**Result: reproduced; the time constant is 0.06 to 0.08 s in every reading, and the SD stays flat after 0.5 s.** Under every reading the time constant is below 0.1 s, and 0.080 to 0.083 s when the average is over all probes. The mean SD after 0.5 s is within 25% of the fitted constant in every reading, and within 25% of the paper's 2.0 spike/s in two. The sizes depend on the spike threshold and the bin time (5.2 to 20.8 spike/s). One is 15.0 (40 SD threshold, bin centres, all probes), but its constant is 1.0 spike/s, not the paper's 2.0.

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first 100-ms
> bin of the PSTH.

Among MGB probes from the seven-fragment design of Figure 2 with three contexts (silence and two
fragments), at least one with the SD over contexts in the first bin at least twice that of every
later bin; the fraction of the 93 probes for which this holds is also given.

In [ ]:
def first_bin(p, key):
    s = p["psth_%s" % key].std(axis=0, ddof=1)
    return s[0] >= 2 * s[1:].max() and s[0] > 0
for key in A.MGB_THRESHOLDS:
    allp = [first_bin(p, key) for g, p in MP]
    des = [(A.cell_id(g["path"]), p["probe"], first_bin(p, key)) for g, p in MP
           if g["n_frag"] == 7 and len(p["contexts"]) == 3 and 0 in p["contexts"]]
    print("threshold %s: %d of %d probes; in the Figure 2 design %d of %d: %s"
          % (key, sum(allp), len(allp), sum(d[2] for d in des), len(des), [d[:2] for d in des if d[2]]))

**Result: reproduced; 4 of the 12 probes in the Figure 2 design have the first-bin pattern under every threshold, and so do 35 to 46 of the 93 probes.** All four come from cell 20071010-ha2-003.

## Blocked claims

- fig5-example-conductances and fig6-conductance-decay: the data set holds no voltage-clamp
  recordings (data check above), so the conductances cannot be computed.
- fig4-strf-performance-low, fig7-example-frequency-context and the fig8 claims: not tested in this
  run for lack of time. The STRF claim needs Wehr's data (in `crcns-ac1.zip`, `wehr/`) and a ridge
  regression over nine windows. The synthetic claims need, for every synthetic file, a map from
  each stimulus to the property varied. Part of that map is in the file notes (for example "2-5,8:
  amplitude variants" in the modulated harmonic tone files), and part has to be read from the
  stimulus descriptions. The synthetic files were opened and cut into probe sets above (cell count
  only).

In [ ]:
print("notebook ran in %.1f min; peak memory %.2f GB" % ((time.time() - T0) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

## Summary

| Claim | Outcome |
|---|---|
| methods-recording-counts | partially reproduced (305 probes, 39 cells, 93 MGB probes and 14 MGB cells match; 132 cells against 123; voltage clamp not in the data) |
| a1-mgb-firing-rates | partially reproduced, on assumed spike detectors (A1 below 1 Hz and MGB evoked above 5 Hz; not all means within 10%) |
| fig2-example-reliability | reproduced (0.31 over cells; the example cell gives 0.61 and 0.74) |
| fig2-example-long-context | reproduced under the amended criterion (no significance from 1.60 to 3.97 s; the strict 2 to 4 s reading fails by 30 ms) |
| fig3a-proportion-significant | reproduced (186 of 305, late fraction 0.252) |
| fig3a-significance-decay | partially reproduced, on the assumption of non-negative sizes |
| fig3b-fractional-power-decay | not reproduced (time constant 1.3 to 1.8 s against 1.04 s) |
| timescale-longer-than-membrane | reproduced |
| fig4-predictability-upper-bound | not reproduced (time constant 1.5 s or longer against 1.04 s) |
| fig4-strf-performance-low | blocked (not tested in this run) |
| fig5-example-conductances | blocked (data missing) |
| fig6-conductance-decay | blocked (data missing) |
| fig7-example-frequency-context | blocked (not tested in this run) |
| fig8a-proportion-by-property | blocked (not tested in this run) |
| fig8b-decay-by-property | blocked (not tested in this run) |
| fig8a-significance-fits | blocked (not tested in this run) |
| fig8c-relative-effects | blocked (not tested in this run) |
| fig8-natural-synthetic-probes-alike | blocked (not tested in this run) |
| fig9-mgb-fast-decay | reproduced |
| fig9-example-first-bin | reproduced |
| figs1-silence-probe | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The Kruskal-Wallis test uses average ranks for ties. The authors' `CxtDep.m` ranks ties by
  order of the trials, which on the quantised traces gives the first contexts lower ranks.
- Details the paper leaves open are taken from the authors' code in the data set: which
  presentations form the trials (the first "Effective repeats" of each sequence), the silence
  probe (first 2.5 s of the intersequence silence), the centring of each trial over the probe
  in the power measure, the PSTH bins and the MGB spike filter.
- A1 spikes are found as excursions above the median-clipped trace, and MGB spikes at three
  thresholds, as the paper gives no detector settings. The rates are over 39 A1 and 14 MGB
  cells, not the paper's 194 and 16.
- Population curves are formed over all probes and over the probes still running; fits use a
  grid of start values. Bounding the sizes at zero is what lets the two-exponential fit of
  Figure 3A describe the decay; with free signs one exponential fits the rise after onset.

What was found about the data set:

- The 42 voltage-clamp recordings (Figures 5 and 6) are not in the data set. Every file is a
  current-clamp or cell-attached trace in mV.
- The data set holds 39 natural sound A1 cells and 14 MGB cells with context files, which give the
  paper's analysed counts exactly (305 and 93 probes). It does not hold the 194 current-clamp
  cells and 16 MGB cells of the firing rates, and 132 A1 cells meet the analysis criterion where
  the paper has 123.
- In 20051107-ha1-001 the trigger labels (printed in the data check) do not match the numbering
  of the stimulus table ("Seq.1" to "Seq.4"); triggers are matched to the table by their fragment
  lists.
- For the synthetic ensembles, the property varied among the conditioning stimuli is written in
  the notes of some files ("2-5,8: amplitude variants") and in the stimulus descriptions of others
  ("TORCs: amplitude variants"). For 19 modulated harmonic tone files and 9 files of transformed
  natural sounds it is not labelled.
- The high-pass filter of `GetSpikes.m` was designed for 4 kHz sampling (its comment) and is applied
  to 10 kHz traces; its automatic threshold (2/3 of the largest value) misses most MGB spikes.